In [35]:
from pathlib import Path
import io
import pymupdf
from html.parser import HTMLParser

In [52]:
class MyHTMLParser(HTMLParser):
    def __init__(self, out: io.TextIOWrapper):
        self._fo = out
        self._tag = None
        super().__init__()
    
    def handle_starttag(self, tag, attrs):
        if tag not in ('p', 'h1', 'h2', 'h3'):
            return
        if tag == self._tag:
            return
        self._tag = tag
        self._fo.write("\n")

    def handle_data(self, data):
        self._fo.write(data)

In [53]:
file = Path('input/test.pdf')
flag = pymupdf.TEXTFLAGS_XHTML & ~pymupdf.TEXT_PRESERVE_IMAGES
doc = pymupdf.open(file)

with open("output/test.txt", "w", -1, "utf-8") as fo, open("output/raw.txt", "w", -1, "utf-8") as fro:
    parser = MyHTMLParser(fo)
    for page in doc.pages():
        txt = page.get_text("xhtml", flags = flag)
        parser.feed(txt)
        fro.write(txt)